# Repeat-seed results

Reads the job logs under `logs/` and builds the summary table averaged over its seeds.

In [1]:
import base64, glob, math, re, statistics, subprocess, tempfile
from collections import defaultdict
from pathlib import Path

from IPython.display import HTML, display
from scipy.stats import sem, t

In [2]:
# Log label -> (scale, decimals shown), in column order. The log has no
# counterexamples line, so parse() counts them.
COLUMNS = {"counterexamples": (1, 1),
           "Total membership queries": (1, 0),
           "Total time (ms)": (1 / 60000, 1),         # shown in minutes
           "Macro precision": (1, 3), "Macro recall": (1, 3),
           "Micro precision": (1, 3), "Micro recall": (1, 3)}

# Log folders are c{N}-nlp-advanced-{model}[-{sampler}], weighted when unsuffixed.
# Model key -> table name, in table order; a model not listed goes last, by key.
MODELS = {"mistral": "mistral-7b-instruct-v0.3",
          "deepseek14b": "DeepSeek-R1-Distill-Qwen-14B",
          "deepseek32b": "DeepSeek-R1-Distill-Qwen-32B"}

CONFIGS = ["C1", "C2", "C3"]


def models_in(results, sampler):
    """The models with a log folder under this sampler, in MODELS order."""
    found = {model for _, model, arm in results if arm == sampler}
    order = list(MODELS)
    return sorted(found, key=lambda m: (order.index(m) if m in order else len(order), m))


def parse(path):
    text = open(path, errors="replace").read()
    config, style, level, model, *arm = Path(path).parent.name.split("-")
    precomp = re.search(r"^Run parameters:.*\bprecomp=(\w+)", text, re.M)
    row = {
        "path": path,
        "config": config.upper(),
        "prompt": f"{style}-{level}",                
        "precomp": precomp[1].capitalize() if precomp else "?",
        "model": model,
        "sampler": arm[0] if arm else "weighted",
        "done": "Ontology learned successfully!" in text,
    }
    for label, (scale, _) in COLUMNS.items():
        hit = re.search(rf"^{re.escape(label)}: ([\d.]+)$", text, re.M)
        row[label] = float(hit[1]) * scale if hit else None
    # Unanchored: vLLM's progress bar ends on a carriage return, so a
    # counterexample announced after one shares its physical line.
    row["counterexamples"] = len(re.findall(r"Counterexample \d+ at sample", text))
    return row

In [3]:
def cell(values, digits):
    """Mean and 95% t interval over the runs that logged this figure."""
    values = [v for v in values if v is not None]
    if not values:
        return "--"
    mean = statistics.mean(values)
    if len(values) < 2:
        # A single run's count prints as logged: 20, not 20.0.
        return str(mean) if isinstance(mean, int) else f"{mean:.{digits}f}"
    ci = t.ppf(0.975, len(values) - 1) * sem(values)
    # Widen until one significant figure survives: a CI shown as 0.0 reads as
    # no spread at all.
    ci_digits = digits if ci <= 0 else max(digits, -math.floor(math.log10(ci)))
    return rf"{mean:.{digits}f}\,{{\scriptsize $\pm${ci:.{ci_digits}f}}}"

In [4]:
def load():
    """-> {(config, model, sampler): [finished runs]}, one run per log.
    Every folder gets a key, so one whose jobs all died shows as dashes."""
    results = defaultdict(list)
    for path in sorted(glob.glob("../logs/**/exactlearner-*.log", recursive=True)):
        row = parse(path)
        runs = results[row["config"], row["model"], row["sampler"]]
        # A run stopped at walltime has no final hypothesis to evaluate.
        if row["done"]:
            runs.append(row)
    return dict(results)

In [5]:
HEADER = r""" &  & \multicolumn{3}{c}{\textbf{Learning cost}} & \multicolumn{4}{c}{\textbf{Learned quality}} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-9}
 &  &  & \textbf{Mem.} & \textbf{Time} & \textbf{Macro} & \textbf{Macro} & \textbf{Micro} & \textbf{Micro} \\
\textbf{Config} & \textbf{n} & \textbf{CEs} & \textbf{queries} & \textbf{(min)} & \textbf{Precision} & \textbf{Recall} & \textbf{Precision} & \textbf{Recall} \\"""

N_COLS = 9


def row_cells(runs):
    """n, then one cell per column."""
    return [str(len(runs))] + [cell([r[label] for r in runs], digits)
                               for label, (_, digits) in COLUMNS.items()]


def table_latex(results, sampler, models, header, row, caption, label):
    """Per model a block of one row per config. row(runs) gives the cells after
    Config; dashes where no run finished or row() returns None."""
    out = [r"\begin{table*}[]", r"\centering", r"\setlength{\tabcolsep}{4pt}",
           r"\begin{tabular}{@{}lrrrrrrrr@{}}", r"\toprule", header, r"\midrule"]
    for i, model in enumerate(models):
        if i:
            out.append(r"\addlinespace")
        out.append(rf"\multicolumn{{{N_COLS}}}{{@{{}}l}}{{\itshape {MODELS.get(model, model)}}} \\")
        for config in CONFIGS:
            runs = results.get((config, model, sampler))
            cells = (row(runs) if runs else None) or ["--"] * (N_COLS - 1)
            out.append(" & ".join([config] + cells) + r" \\")
    out += [r"\bottomrule", r"\end{tabular}", rf"\caption{{{caption}}}",
            rf"\label{{{label}}}", r"\end{table*}"]
    return "\n".join(out)


def summary_latex(results, sampler="weighted", models=None):
    """models: folder keys to show, in order; default every model with a folder."""
    models = models or models_in(results, sampler)
    # Caption read off the runs shown: "NLP-advanced, precomp=False, weighted sampler".
    shown = [r for m in models for c in CONFIGS for r in results.get((c, m, sampler), [])]
    prompt, precomp = (", ".join(sorted({r[k] for r in shown})) for k in ("prompt", "precomp"))
    # The weighted table keeps the bare label the paper already cites.
    label = "table:repeats" if sampler == "weighted" else f"table:repeats-{sampler}"
    return table_latex(results, sampler, models, HEADER, row_cells,
                       f"{prompt}, precomp={precomp}, {sampler} sampler", label)

In [6]:
PREAMBLE = r"""\documentclass[border=6pt,varwidth=40cm]{standalone}
\usepackage{booktabs,caption,threeparttable}
% standalone cannot hold a float; threeparttable sets the caption as wide as the table.
\renewenvironment{table*}[1][]{\begin{threeparttable}}{\end{threeparttable}}
\begin{document}
"""

ZOOM_FACTOR = 1.5  # one factor for every table, so type matches across them


def show(src):
    """Render the table as SVG; print the LaTeX if it does not compile."""
    with tempfile.TemporaryDirectory() as tmp:
        (Path(tmp) / "t.tex").write_text(PREAMBLE + src + "\n\\end{document}\n")
        ok = lambda *cmd: subprocess.run(cmd, cwd=tmp, capture_output=True).returncode == 0
        # --no-fonts draws glyphs as paths: browsers do not render SVG fonts.
        if (ok("latex", "-interaction=nonstopmode", "-halt-on-error", "t.tex")
                and ok("dvisvgm", "--exact-bbox", "--bbox=4pt", "--no-fonts",
                       f"--zoom={ZOOM_FACTOR}", "t.dvi", "-o", "t.svg")):
            svg = base64.b64encode((Path(tmp) / "t.svg").read_bytes()).decode()
            # An <img> keeps each SVG's glyph ids apart; white keeps dark themes legible.
            display(HTML(f'<img src="data:image/svg+xml;base64,{svg}" style="background:#fff">'))
            return
    print(src)

## Results

In [7]:
results = load()

In [8]:
mistral_weighted_summary = summary_latex(results, sampler="weighted", models=["mistral"])
show(mistral_weighted_summary)
#print(mistral_weighted_summary)

## Unweighted sampler

In [9]:
mistral_unweighted_summary = summary_latex(results, sampler="unweighted", models=["mistral"])
show(mistral_unweighted_summary)
#print(mistral_unweighted_summary)

## Counterexample arrival rate

Whether counterexamples arrive more slowly later in a run. The gap is the samples
drawn since the previous one; each seed is one replicate, so the trend test is a
per-seed rank correlation with the counterexample index and a sign test over the
seeds. A run submitted on its own is one replicate, at n=1. Weighted arm only; pass
`sampler="unweighted"` to `arrival_latex` for the other one.

In [10]:
CE_LINE = re.compile(r"Counterexample \d+ at sample (\d+) \(\+(\d+) since the last one\)")
BUDGET = re.compile(r"PAC sample budget \(numberOfSamples\) = (\d+)")

QUARTERS = 4


def arrivals(path):
    """-> (gaps between counterexamples, sample of the last one, budget)."""
    text = open(path, errors="replace").read()
    ces = [(int(s), int(g)) for s, g in CE_LINE.findall(text)]
    budget = BUDGET.search(text)
    return ([g for _, g in ces], ces[-1][0] if ces else 0,
            int(budget[1]) if budget else None)





In [11]:
def arrival_cells(runs):
    """n, median CEs, mean gap per quarter pooled over seeds, Q4/Q1 and tail."""
    runs = [arrivals(r["path"]) for r in runs]
    runs = [(g, last, budget) for g, last, budget in runs if len(g) >= QUARTERS]
    if not runs:
        return None

    quarters = []
    for q in range(QUARTERS):
        pooled = [g[len(g) * q // QUARTERS:len(g) * (q + 1) // QUARTERS] for g, _, _ in runs]
        quarters.append(statistics.mean([x for chunk in pooled for x in chunk]))

    # Budget left unspent after the last counterexample: high means the run ran
    # out of things to learn long before it ran out of budget.
    tail = statistics.median([(b - last) / b for _, last, b in runs if b])
    ces = statistics.median([len(g) for g, _, _ in runs])
    return ([str(len(runs)), f"{ces:.0f}"] + [f"{q:.0f}" for q in quarters]
            + [f"{quarters[-1] / quarters[0]:.2f}", f"{tail * 100:.0f}\\%"])


ARRIVAL_HEADER = r""" &  &  & \multicolumn{4}{c}{\textbf{Mean gap}} & \multicolumn{2}{c}{\textbf{Trend}} \\
\cmidrule(lr){4-7} \cmidrule(lr){8-9}
\textbf{Config} & \textbf{n} & \textbf{CEs} & \textbf{Q1} & \textbf{Q2} & \textbf{Q3} & \textbf{Q4} & \textbf{Q4/Q1} & \textbf{tail} \\"""


def arrival_latex(results, sampler="weighted", models=None):
    """models: folder keys to show, in order; default every model with a folder."""
    return table_latex(results, sampler, models or models_in(results, sampler),
                       ARRIVAL_HEADER, arrival_cells,
                       "Mean samples between counterexamples, by quarter of each run, "
                       "pooled over the replicates. Tail is the unspent budget. "
                       "A gap cannot exceed the budget left, biasing late quarters down.",
                       "table:arrivals")


In [12]:
arrival_tex = arrival_latex(results, models=["mistral"])
show(arrival_tex)